# **Global Seismic Trends: Data-Driven Earthquake Insights**

**Problem Statement**

# **Step 1: Data Preparation**

**Step 1.1: Defined the USGS API endpoint & parameters**

In [1]:
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

In [2]:
import requests
import pandas as pd
from datetime import datetime

**Step 1.2: Handled API limits / large-result errors and Fetched data month-by-month and collected responses**

In [3]:
import requests
import pandas as pd
from datetime import datetime, timedelta

def fetch_monthly_data(start_date, end_date, minmag=4.5):
    """Fetch earthquake data month by month to avoid USGS 20k limit."""
    
    url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    all_events = []

    # Convert to datetime objects
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end   = datetime.strptime(end_date, "%Y-%m-%d")

    # Loop month-by-month
    current = start
    while current < end:
        # Month start & next month start
        month_start = current
        month_end = (current.replace(day=28) + timedelta(days=4)).replace(day=1)

        # Don't exceed final date
        if month_end > end:
            month_end = end

        print(f"Fetching: {month_start.date()} → {month_end.date()}")

        params = {
            "format": "geojson",
            "starttime": month_start.strftime("%Y-%m-%d"),
            "endtime": month_end.strftime("%Y-%m-%d"),
            "minmagnitude": minmag
        }

        headers = {"User-Agent": "Mozilla/5.0"}

        # Request with timeout & browser headers
        response = requests.get(
            url,
            params=params,
            timeout=60,
            headers=headers
        )

        # Raise if error
        response.raise_for_status()

        data = response.json()
        events = data.get("features", [])

        print(f" → Retrieved {len(events)} events")

        all_events.extend(events)

        # Move to next month
        current = month_end

    # Convert into DataFrame
    df = pd.json_normalize(all_events)
    return df


**Step1.3: Converted the GeoJSON to a DataFrame**

In [4]:
df = fetch_monthly_data("2020-01-01", "2025-01-01", minmag=4.5)


Fetching: 2020-01-01 → 2020-02-01
 → Retrieved 656 events
Fetching: 2020-02-01 → 2020-03-01
 → Retrieved 516 events
Fetching: 2020-03-01 → 2020-04-01
 → Retrieved 496 events
Fetching: 2020-04-01 → 2020-05-01
 → Retrieved 456 events
Fetching: 2020-05-01 → 2020-06-01
 → Retrieved 557 events
Fetching: 2020-06-01 → 2020-07-01
 → Retrieved 586 events
Fetching: 2020-07-01 → 2020-08-01
 → Retrieved 550 events
Fetching: 2020-08-01 → 2020-09-01
 → Retrieved 527 events
Fetching: 2020-09-01 → 2020-10-01
 → Retrieved 556 events
Fetching: 2020-10-01 → 2020-11-01
 → Retrieved 596 events
Fetching: 2020-11-01 → 2020-12-01
 → Retrieved 471 events
Fetching: 2020-12-01 → 2021-01-01
 → Retrieved 527 events
Fetching: 2021-01-01 → 2021-02-01
 → Retrieved 575 events
Fetching: 2021-02-01 → 2021-03-01
 → Retrieved 846 events
Fetching: 2021-03-01 → 2021-04-01
 → Retrieved 1258 events
Fetching: 2021-04-01 → 2021-05-01
 → Retrieved 599 events
Fetching: 2021-05-01 → 2021-06-01
 → Retrieved 624 events
Fetching: 202

In [ ]:
df

**Step 1.4: Renamed / flattened nested columns**

In [6]:
 # Rename important columns
df = df.rename(columns={
            "properties.mag": "magnitude",
            "properties.place": "place",
            "properties.time": "time",
            "geometry.coordinates": "coordinates"
        })

In [7]:
df.columns = df.columns.str.replace(".", "_", regex=False)

In [8]:
df.columns

Index(['type', 'id', 'magnitude', 'place', 'time', 'properties_updated',
       'properties_tz', 'properties_url', 'properties_detail',
       'properties_felt', 'properties_cdi', 'properties_mmi',
       'properties_alert', 'properties_status', 'properties_tsunami',
       'properties_sig', 'properties_net', 'properties_code', 'properties_ids',
       'properties_sources', 'properties_types', 'properties_nst',
       'properties_dmin', 'properties_rms', 'properties_gap',
       'properties_magType', 'properties_type', 'properties_title',
       'geometry_type', 'coordinates'],
      dtype='object')

**Step1.5: Extracted coordinates into numeric columns**

In [9]:
# Extract lon, lat, depth
df["longitude"] = df["coordinates"].apply(lambda x: x[0])
df["latitude"] = df["coordinates"].apply(lambda x: x[1])
df["depth_km"] = df["coordinates"].apply(lambda x: x[2])

df = df.drop(columns=["coordinates"])

**Step 1.6: Converted timestamp to datetime**

In [10]:
# Convert timestamp
df["time"] = pd.to_datetime(df["time"], unit="ms")

In [ ]:
df.head()

**Step 1.7: Checked for nulls & empty strings**

In [11]:
df.isnull().sum()

type                      0
id                        0
magnitude                 0
place                     0
time                      0
properties_updated        0
properties_tz         37251
properties_url            0
properties_detail         0
properties_felt       29837
properties_cdi        29837
properties_mmi        33296
properties_alert      33705
properties_status         0
properties_tsunami        0
properties_sig            0
properties_net            0
properties_code           0
properties_ids            0
properties_sources        0
properties_types          0
properties_nst        18089
properties_dmin         170
properties_rms            0
properties_gap          154
properties_magType        0
properties_type           0
properties_title          0
geometry_type             0
longitude                 0
latitude                  0
depth_km                  0
dtype: int64

In [12]:
df['properties_tz'].value_counts(normalize=True)

Series([], Name: proportion, dtype: float64)

In [13]:
df=df.drop(columns=['properties_tz'])

In [ ]:
df

**Step 1.8: Extracted country from place with regex / rules**

In [14]:
import re

def extract_country(place):
    if pd.isna(place):
        return "Unknown"

    # If comma exists, country is after the last comma
    if "," in place:
        return place.split(",")[-1].strip()

    # Otherwise, try extracting last word (but may not be accurate)
    # Example: "Off the coast of Chile"
    match = re.findall(r"[A-Za-z]+$", place)
    return match[0] if match else "Unknown"

# Apply to dataset
df["country"] = df["place"].apply(extract_country)

**Step 1.9: Cleaned text/string columns**

In [15]:
string_cols = [
    "id","place", "properties_alert", "properties_status",
    "properties_net", "properties_ids","properties_sources", "properties_types",
    "properties_magType", "properties_type", "properties_title","country"
]

for col in string_cols:
    if col in df.columns:  # check column exists
        df[col] = (
            df[col]
            .astype("string")                  # convert column to string dtype
            .fillna("")                        # fill missing values
            .str.strip()  
            .str.lstrip(",") 
            .str.rstrip(",")                                                                                    # remove leading/trailing spaces
            .str.replace(r"\s+", " ", regex=True)  # normalize multi spaces
            .str.lower()                       # convert to lowercase
        )

**Step 1.10: Converted numeric fields to numeric types safely**

In [16]:
numeric_cols = [
    "magnitude", "depth_km",
    "properties_nst", "properties_dmin", "properties_rms",
    "properties_gap", "properties_sig",
    "properties_magError", "properties_depthError",
    "properties_magNst"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [17]:
df.dtypes

type                          object
id                    string[python]
magnitude                    float64
place                 string[python]
time                  datetime64[ns]
properties_updated             int64
properties_url                object
properties_detail             object
properties_felt              float64
properties_cdi               float64
properties_mmi               float64
properties_alert      string[python]
properties_status     string[python]
properties_tsunami             int64
properties_sig                 int64
properties_net        string[python]
properties_code               object
properties_ids        string[python]
properties_sources    string[python]
properties_types      string[python]
properties_nst               float64
properties_dmin              float64
properties_rms               float64
properties_gap               float64
properties_magType    string[python]
properties_type       string[python]
properties_title      string[python]
g

**Step 1.11: Imputed missing numeric values with reasoned choices**

In [18]:
df.isnull().sum()

type                      0
id                        0
magnitude                 0
place                     0
time                      0
properties_updated        0
properties_url            0
properties_detail         0
properties_felt       29837
properties_cdi        29837
properties_mmi        33296
properties_alert          0
properties_status         0
properties_tsunami        0
properties_sig            0
properties_net            0
properties_code           0
properties_ids            0
properties_sources        0
properties_types          0
properties_nst        18089
properties_dmin         170
properties_rms            0
properties_gap          154
properties_magType        0
properties_type           0
properties_title          0
geometry_type             0
longitude                 0
latitude                  0
depth_km                  0
country                   0
dtype: int64

In [19]:
df['properties_alert'] = ( df['properties_alert'].fillna('Not_attack').astype(str).str.strip().str.lower())

In [20]:
import numpy as np

df['properties_alert'] = (
    df['properties_alert']
    .replace(["", " ", "  ", "None", None], np.nan)  # convert empty/space/None → NaN
    .fillna("none")                                   # finally fill with 'none'
)


In [21]:
df['properties_alert'].value_counts()

properties_alert
none      33705
green      3387
yellow      120
orange       22
red          17
Name: count, dtype: int64

**properties_felt, properties_cdi, properties_mmi → fill 0 (no reports / no measurement).**

In [22]:
for col in ['properties_felt','properties_cdi','properties_mmi']:
  if col in df.columns:
    df[col] = df[col].fillna(0)

**properties_nst, properties_dmin, properties_gap → fill with median**

In [23]:
for col in ['properties_nst','properties_dmin','properties_gap']:
  if col in df.columns:
    df[col] = df[col].fillna(df[col].median())

In [24]:
df.isnull().sum()

type                  0
id                    0
magnitude             0
place                 0
time                  0
properties_updated    0
properties_url        0
properties_detail     0
properties_felt       0
properties_cdi        0
properties_mmi        0
properties_alert      0
properties_status     0
properties_tsunami    0
properties_sig        0
properties_net        0
properties_code       0
properties_ids        0
properties_sources    0
properties_types      0
properties_nst        0
properties_dmin       0
properties_rms        0
properties_gap        0
properties_magType    0
properties_type       0
properties_title      0
geometry_type         0
longitude             0
latitude              0
depth_km              0
country               0
dtype: int64

In [25]:
df.shape

(37251, 32)

**Step 1.12: Add Derived columns**

Date columns

In [26]:
df['time'] = pd.to_datetime(df['time'])

df['year'] = df['time'].dt.year
df['month'] = df['time'].dt.month
df['day'] = df['time'].dt.day
df['day_of_week'] = df['time'].dt.day_name()
df['hour'] = df['time'].dt.hour


In [27]:
# Convert hour to numeric, coerce errors to NaN
df['hour'] = pd.to_numeric(df['hour'], errors='coerce')

# Replace NaN with None (so MySQL accepts NULL)
df['hour'] = df['hour'].where(df['hour'].notnull(), None)



In [28]:
print("🔹 Magnitude Distribution (Integer Groupby)")
print(df.groupby(df['magnitude'].astype(int)).size())


🔹 Magnitude Distribution (Integer Groupby)
magnitude
4    28592
5     8008
6      583
7       65
8        3
dtype: int64


In [29]:
print("\n🔹 Strong Earthquakes (mag ≥ 6.0)")
print(df[df['magnitude'] >= 6].groupby(df['magnitude'].astype(int)).size())

print("\n🔹 Destructive Earthquakes (mag ≥ 7.0)")
print(df[df['magnitude'] >= 7].groupby(df['magnitude'].astype(int)).size())



🔹 Strong Earthquakes (mag ≥ 6.0)
magnitude
6    583
7     65
8      3
dtype: int64

🔹 Destructive Earthquakes (mag ≥ 7.0)
magnitude
7    65
8     3
dtype: int64


Depth Category (Based on rule: shallow < 70 km)

In [30]:
df['depth_category'] = df['depth_km'].apply(
    lambda x: 'Shallow' if x < 70 else 'Deep'
)


Magnitude Category (Strong / Destructive)

In [31]:
def classify_mag(m):
    if m >= 7.0:
        return 'Destructive'
    elif m >= 6.0:
        return 'Strong'
    else:
        return 'Normal'

df['mag_category'] = df['magnitude'].apply(classify_mag)


In [32]:
df.shape

(37251, 39)

In [33]:
df.columns

Index(['type', 'id', 'magnitude', 'place', 'time', 'properties_updated',
       'properties_url', 'properties_detail', 'properties_felt',
       'properties_cdi', 'properties_mmi', 'properties_alert',
       'properties_status', 'properties_tsunami', 'properties_sig',
       'properties_net', 'properties_code', 'properties_ids',
       'properties_sources', 'properties_types', 'properties_nst',
       'properties_dmin', 'properties_rms', 'properties_gap',
       'properties_magType', 'properties_type', 'properties_title',
       'geometry_type', 'longitude', 'latitude', 'depth_km', 'country', 'year',
       'month', 'day', 'day_of_week', 'hour', 'depth_category',
       'mag_category'],
      dtype='object')

In [34]:
df = df.drop(columns=['properties_url', 'properties_detail', 'type', 'properties_title','geometry_type'])


In [35]:
continent_map = {
    # Asia
    "indonesia": "Asia",
    "philippines": "Asia",
    "japan": "Asia",
    "japan region": "Asia",
    "china": "Asia",
    "taiwan": "Asia",
    "india": "Asia",
    "india region": "Asia",
    "turkey": "Asia",
    "iran": "Asia",
    "afghanistan": "Asia",
    "myanmar": "Asia",
    "pakistan": "Asia",
    "nepal": "Asia",
    "bangladesh": "Asia",
    "laos": "Asia",
    "syria": "Asia",
    "cyprus": "Asia",
    "vietnam": "Asia",
    "armenia": "Asia",
    "azerbaijan": "Asia",
    "saudi arabia": "Asia",
    "iraq": "Asia",
    "kazakhstan": "Asia",
    "uzbekistan": "Asia",
    "turkmenistan": "Asia",
    "kyrgyzstan": "Asia",
    "tajikistan": "Asia",
    "malaysia": "Asia",
    "thailand": "Asia",
    "israel": "Asia",
    "yemen": "Asia",
    "japan earthquake": "Asia",
    "kahramanmaras earthquake sequence": "Asia",
    "north korea": "Asia",
    "south korea": "Asia",
    "bhutan": "Asia",
    "korea": "Asia",
    "xizang": "Asia",
    
    # Oceania
    "tonga": "Oceania",
    "papua new guinea": "Oceania",
    "vanuatu": "Oceania",
    "new zealand": "Oceania",
    "new zealand region": "Oceania",
    "australia": "Oceania",
    "australia region": "Oceania",
    "fiji": "Oceania",
    "micronesia": "Oceania",
    "federated states of micronesia": "Oceania",
    "solomon islands": "Oceania",
    "hawaii": "Oceania",
    "guam": "Oceania",
    "northern mariana islands": "Oceania",
    "american samoa": "Oceania",
    "kiribati region": "Oceania",
    "samoa": "Oceania",
    "new caledonia": "Oceania",
    "wallis and futuna": "Oceania",
    "french polynesia region": "Oceania",
    "tuvalu": "Oceania",

    # North America
    "alaska": "North America",
    "canada": "North America",
    "canada region": "North America",
    "united states": "North America",
    "america": "North America",
    "mexico": "North America",
    "cuba": "North America",
    "guatemala": "North America",
    "el salvador": "North America",
    "honduras": "North America",
    "nicaragua": "North America",
    "panama": "North America",
    "haiti": "North America",
    "dominican republic": "North America",
    "puerto rico": "North America",
    "u.s. virgin islands": "North America",
    "antigua and barbuda": "North America",
    "bahamas": "North America",
    "barbados": "North America",
    "cayman islands": "North America",
    "grenada": "North America",
    "jamaica": "North America",
    "saint kitts and nevis": "North America",
    "saint pierre and miquelon": "North America",

    # South America
    "chile": "South America",
    "argentina": "South America",
    "peru": "South America",
    "colombia": "South America",
    "ecuador": "South America",
    "ecuador region": "South America",
    "venezuela": "South America",
    "bolivia": "South America",
    "brazil": "South America",
    "guyana": "South America",
    "suriname": "South America",

    # Europe
    "greece": "Europe",
    "portugal": "Europe",
    "portugal region": "Europe",
    "russia": "Europe",
    "russia region": "Europe",
    "iceland": "Europe",
    "italy": "Europe",
    "albania": "Europe",
    "romania": "Europe",
    "croatia": "Europe",
    "cyprus": "Europe",
    "spain": "Europe",
    "poland": "Europe",
    "bosnia and herzegovina": "Europe",
    "slovakia": "Europe",
    "norway": "Europe",
    "sweden": "Europe",
    "france": "Europe",
    "gibraltar": "Europe",
    "montenegro": "Europe",

    # Africa
    "somalia": "Africa",
    "tanzania": "Africa",
    "ethiopia": "Africa",
    "uganda": "Africa",
    "eritrea": "Africa",
    "zambia": "Africa",
    "zimbabwe": "Africa",
    "mozambique": "Africa",
    "south africa": "Africa",
    "democratic republic of the congo": "Africa",
    "ghana": "Africa",
    "kenya": "Africa",
    "rwanda": "Africa",
    "gabon": "Africa",
    "algeria": "Africa",
    "morocco": "Africa",
    "libya": "Africa",
    "egypt": "Africa",
    "tunisia": "Africa",
    "mauritius": "Africa",
    "mayotte": "Africa",
    "madagascar": "Africa",
    "angola": "Africa",
    "botswana": "Africa",
    "congo-uganda": "Africa",

    # Middle East
    "qatar": "Asia",
    "kuwait": "Asia",
    "oman": "Asia",
}


In [36]:
df['continent'] = df['country'].str.lower().map(continent_map)


In [37]:
import numpy as np

df['continent'] = (
    df['continent']
    .replace(["", " ", "  ", "None", None,"nan"], np.nan)  # convert empty/space/None → NaN
    .fillna("Unknown")                                   # finally fill with 'none'
)

In [38]:
df['continent'].unique()

array(['Asia', 'South America', 'Unknown', 'North America', 'Oceania',
       'Europe', 'Africa'], dtype=object)

In [39]:
df.shape

(37251, 35)

In [40]:
df.dtypes

id                    string[python]
magnitude                    float64
place                 string[python]
time                  datetime64[ns]
properties_updated             int64
properties_felt              float64
properties_cdi               float64
properties_mmi               float64
properties_alert              object
properties_status     string[python]
properties_tsunami             int64
properties_sig                 int64
properties_net        string[python]
properties_code               object
properties_ids        string[python]
properties_sources    string[python]
properties_types      string[python]
properties_nst               float64
properties_dmin              float64
properties_rms               float64
properties_gap               float64
properties_magType    string[python]
properties_type       string[python]
longitude                    float64
latitude                     float64
depth_km                     float64
country               string[python]
y

In [41]:
df.columns

Index(['id', 'magnitude', 'place', 'time', 'properties_updated',
       'properties_felt', 'properties_cdi', 'properties_mmi',
       'properties_alert', 'properties_status', 'properties_tsunami',
       'properties_sig', 'properties_net', 'properties_code', 'properties_ids',
       'properties_sources', 'properties_types', 'properties_nst',
       'properties_dmin', 'properties_rms', 'properties_gap',
       'properties_magType', 'properties_type', 'longitude', 'latitude',
       'depth_km', 'country', 'year', 'month', 'day', 'day_of_week', 'hour',
       'depth_category', 'mag_category', 'continent'],
      dtype='object')

# **Step 2: SQL Connection**

**Step 2.1: Store Cleaned Earthquake Data in MySQL**

**Using MySQL Connector**

In [106]:
import mysql.connector

conn= mysql.connector.connect(
    host="localhost",
    user="root",
    password="root"
)
cursor = conn.cursor()
print("MySQL connection established!")



MySQL connection established!


In [ ]:
cursor.execute("DROP TABLE IF EXISTS earthquake;")
conn.commit()

print("Table 'earthquakes' deleted successfully!")

cursor.close()
conn.close()

**Create a Database in MySQL**

In [108]:
cursor.execute("CREATE DATABASE IF NOT EXISTS earthquake_db;")
print("MySQL database 'earthquake_db' created successfully!")

MySQL database 'earthquake_db' created successfully!


**Create a Table in MySQL**

In [109]:
cursor.execute("USE earthquake_db;")

create_table = """
CREATE TABLE IF NOT EXISTS earthquake (
    id VARCHAR(50),
    magnitude FLOAT,
    place TEXT,
    time DATETIME,
    properties_updated BIGINT,
    properties_felt FLOAT,
    properties_cdi FLOAT,
    properties_mmi FLOAT,
    properties_alert VARCHAR(20),
    properties_status VARCHAR(20),
    properties_tsunami INT,
    properties_sig INT,
    properties_net VARCHAR(10),
    properties_code VARCHAR(20),
    properties_ids TEXT,
    properties_sources TEXT,
    properties_types TEXT,
    properties_nst FLOAT,
    properties_dmin FLOAT,
    properties_rms FLOAT,
    properties_gap FLOAT,
    properties_magType VARCHAR(20),
    properties_type VARCHAR(50),
    longitude FLOAT,
    latitude FLOAT,
    depth_km FLOAT,
    country VARCHAR(50),
    year INT,
    month INT,
    day INT,
    day_of_week VARCHAR(20),
    hour INT,
    depth_category VARCHAR(20),
    mag_category VARCHAR(20),
    continent VARCHAR(50)
);
"""

cursor.execute(create_table)
conn.commit()
print("Table created!")



Table created!


**Insert DataFrame into MySQL (mysql.connector method)**

In [110]:
insert_query = """
INSERT INTO earthquake (
    id, magnitude, place, time, properties_updated,
    properties_felt, properties_cdi, properties_mmi,
    properties_alert, properties_status, properties_tsunami,
    properties_sig, properties_net, properties_code, properties_ids,
    properties_sources, properties_types, properties_nst,
    properties_dmin, properties_rms, properties_gap,
    properties_magType, properties_type, longitude, latitude,
    depth_km, country, year, month, day, day_of_week,
    hour, depth_category, mag_category, continent
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s,%s,%s)
"""



In [61]:
print("Number of SQL placeholders:", insert_query.count("%s"))


Number of SQL placeholders: 35


In [62]:
for _, row in df.iterrows():
    cursor.execute(insert_query, tuple(row))

conn.commit()
print("Data inserted successfully!")


Data inserted successfully!


# **Step 3: Analyst Tasks**

**Step 3.1: Magnitude & Depth**

1. Top 10 strongest earthquakes (mag).

In [111]:
query_1= """
SELECT id, magnitude, place, time, depth_km, country
FROM earthquake
ORDER BY magnitude DESC
LIMIT 10;
"""

cursor.execute(query_1)
rows = cursor.fetchall()
cols = [desc[0] for desc in cursor.description]

df_top10 = pd.DataFrame(rows, columns=cols)
df_top10


,id,magnitude,place,time,depth_km,country
0,ak0219neiszm,8.2,"2021 chignik, alaska earthquake",2021-07-29 06:15:49,35.000,alaska earthquake
1,us6000f53e,8.1,2021 south sandwich islands earthquake,2021-08-12 18:35:17,22.790,earthquake
2,us7000dflf,8.1,"2021 kermadec islands, new zealand earthquake",2021-03-04 19:28:33,28.930,new zealand earthquake
3,us6000jllz,7.8,"pazarcik earthquake, kahramanmaras earthquake ...",2023-02-06 01:17:34,10.000,kahramanmaras earthquake sequence
4,us7000asvb,7.8,"2020 perryville, alaska earthquake",2020-07-22 06:12:45,28.000,alaska earthquake
5,us6000dg77,7.7,southeast of the loyalty islands,2021-02-10 13:19:56,10.000,islands
6,us60007idc,7.7,"123 km nnw of lucea, jamaica",2020-01-28 19:10:25,14.860,jamaica
7,us6000kd0n,7.7,southeast of the loyalty islands,2023-05-19 02:57:03,18.053,islands
8,us6000kawn,7.6,"82 km wnw of hihifo, tonga",2023-05-10 16:02:00,210.000,tonga
9,us6000c9hg,7.6,"2020 sand point, alaska earthquake",2020-10-19 20:54:39,28.370,alaska earthquake


2. Top 10 deepest earthquakes (depth_km).


In [64]:
query_2= """
SELECT id, magnitude, place, time, depth_km, country
FROM earthquake
ORDER BY depth_km DESC
LIMIT 10;
"""

cursor.execute(query_2)
rows = cursor.fetchall()
cols = [desc[0] for desc in cursor.description]

df_top10 = pd.DataFrame(rows, columns=cols)
df_top10

,id,magnitude,place,time,depth_km,country
0,us6000dhfx,4.5,south of the fiji islands,2021-02-13 16:26:42,664.740,islands
1,us7000ingi,7.0,south of the fiji islands,2022-11-09 09:51:04,660.000,islands
2,us7000inmi,4.9,south of the fiji islands,2022-11-09 18:41:05,656.429,islands
3,us6000fwrg,4.5,fiji region,2021-10-22 14:21:03,654.810,region
4,us6000ngkc,4.5,south of the fiji islands,2024-07-29 00:59:46,653.779,islands
5,us6000kq6n,4.7,"209 km se of levuka, fiji",2023-07-05 09:56:52,653.516,fiji
6,us7000inmb,5.1,south of the fiji islands,2022-11-09 17:49:46,650.921,islands
7,us7000kvfm,5.0,"100 km nw of batang, indonesia",2023-09-13 05:34:33,650.655,indonesia
8,us7000bf0u,4.7,"210 km n of palu, indonesia",2020-08-30 12:01:02,646.800,indonesia
9,us6000kxlu,4.5,"228 km nne of palu, indonesia",2023-08-03 18:05:53,646.537,indonesia


3. Shallow earthquakes < 50 km and mag > 7.5.


In [65]:
query_3= """
SELECT 
    id,
    magnitude,
    place,
    time,
    depth_km,
    depth_category,
    country
FROM earthquake
WHERE depth_km < 50
  AND magnitude > 7.5
ORDER BY magnitude DESC;
"""

cursor.execute(query_3)
rows = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]

df_shallow_highmag = pd.DataFrame(rows, columns=columns)
df_shallow_highmag


,id,magnitude,place,time,depth_km,depth_category,country
0,ak0219neiszm,8.2,"2021 chignik, alaska earthquake",2021-07-29 06:15:49,35.000,Shallow,alaska earthquake
1,us7000dflf,8.1,"2021 kermadec islands, new zealand earthquake",2021-03-04 19:28:33,28.930,Shallow,new zealand earthquake
2,us6000f53e,8.1,2021 south sandwich islands earthquake,2021-08-12 18:35:17,22.790,Shallow,earthquake
3,us7000asvb,7.8,"2020 perryville, alaska earthquake",2020-07-22 06:12:45,28.000,Shallow,alaska earthquake
4,us6000jllz,7.8,"pazarcik earthquake, kahramanmaras earthquake ...",2023-02-06 01:17:34,10.000,Shallow,kahramanmaras earthquake sequence
5,us60007idc,7.7,"123 km nnw of lucea, jamaica",2020-01-28 19:10:25,14.860,Shallow,jamaica
6,us6000dg77,7.7,southeast of the loyalty islands,2021-02-10 13:19:56,10.000,Shallow,islands
7,us6000kd0n,7.7,southeast of the loyalty islands,2023-05-19 02:57:03,18.053,Shallow,islands
8,us6000c9hg,7.6,"2020 sand point, alaska earthquake",2020-10-19 20:54:39,28.370,Shallow,alaska earthquake
9,us7000i9bw,7.6,"35 km ssw of aguililla, mexico",2022-09-19 18:05:08,26.943,Shallow,mexico


4. Average depth per continent.

In [66]:
query = """
SELECT continent, avg(depth_km) AS avg_depth
FROM earthquake
GROUP BY continent
ORDER BY avg_depth DESC;
"""

cursor.execute(query)
df_continent = pd.DataFrame(cursor.fetchall(), columns=[column[0] for column in cursor.description])
df_continent



,continent,avg_depth
0,South America,88.678459
1,Oceania,73.453560
2,Unknown,63.839985
3,Asia,45.880896
4,Europe,40.591502
5,North America,35.742975
6,Africa,9.887054


5. Average magnitude per magnitude type (magType).

In [67]:
query = """
SELECT properties_magType AS magType,
       AVG(magnitude) AS avg_magnitude,
       COUNT(*) AS total_events
FROM earthquake
GROUP BY properties_magType
ORDER BY avg_magnitude DESC;
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype


,magType,avg_magnitude,total_events
0,mwc,6.150000,2
1,mwb,5.812500,24
2,ms_20,5.800000,1
3,mwp,5.418750,8
4,mww,5.383732,6094
5,ml(texnet),5.200000,1
6,mw,5.005942,69
7,ml,4.794138,174
8,mb,4.684725,29872
9,md,4.670000,8


Step 3.2: Time Analysis

6. Year with most earthquakes.

In [68]:
query = """
SELECT year,count(*) as Total_earthquakes from earthquake
group by year 
order by Total_earthquakes desc
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,year,Total_earthquakes
0,2021,8943
1,2022,7766
2,2023,7651
3,2020,6494
4,2024,6397


7. Month with highest number of earthquakes.

In [69]:
query = """
SELECT month,count(*) as Total_earthquakes from earthquake
group by month 
order by Total_earthquakes desc
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,month,Total_earthquakes
0,3,3589
1,12,3580
2,8,3328
3,1,3270
4,2,3178
5,9,3060
6,10,2954
7,4,2951
8,5,2925
9,11,2899


8. Day of week with most earthquakes.

In [70]:
query = """
SELECT day_of_week,count(*) as Total_earthquakes from earthquake
group by day_of_week
order by Total_earthquakes desc
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,day_of_week,Total_earthquakes
0,Friday,5445
1,Monday,5362
2,Tuesday,5359
3,Sunday,5354
4,Thursday,5327
5,Saturday,5267
6,Wednesday,5137


9. Count of earthquakes per hour of day.

In [71]:
query = """
SELECT hour,count(*) as Total_earthquakes from earthquake
group by hour
order by Total_earthquakes desc
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,hour,Total_earthquakes
0,15,1724
1,3,1696
2,16,1660
3,13,1648
4,21,1647
5,1,1631
6,4,1625
7,2,1615
8,14,1596
9,18,1588


10.   Most active reporting network (net).

In [72]:
query = """
SELECT properties_net, COUNT(*) AS total_reports
FROM earthquake
GROUP BY properties_net
ORDER BY total_reports DESC
LIMIT 5;
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,properties_net,total_reports
0,us,36897
1,ak,153
2,pr,64
3,nc,36
4,ci,32


Step 3.3: Casualties & Economic Loss

11.  Top 5 places with highest casualties.

In [73]:
query = """
SELECT place, SUM(properties_felt) AS casualties
FROM earthquake
GROUP BY place
ORDER BY casualties DESC
limit 5
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,place,casualties
0,"2024 tewksbury, new jersey earthquake",184649.0
1,"4 km se of sparta, north carolina",66717.0
2,"8 km nw of prague, oklahoma",26480.0
3,"5 km nne of magna, utah",26370.0
4,"antelope valley, ca",25986.0


12.  Total estimated economic loss per continent.

In [74]:
query = """
SELECT continent, SUM(properties_sig) AS total_economic_loss
FROM earthquake
GROUP BY continent
ORDER BY total_economic_loss DESC;
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,continent,total_economic_loss
0,Unknown,4243567
1,Asia,4139617
2,Oceania,2785987
3,South America,880258
4,North America,792099
5,Europe,502094
6,Africa,124971


13.  Average economic loss by alert level.


In [75]:
query = """
SELECT properties_alert, avg(properties_sig) AS total_economic_loss
FROM earthquake
GROUP BY properties_alert
ORDER BY total_economic_loss DESC;
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,properties_alert,total_economic_loss
0,red,2308.1176
1,orange,1281.0000
2,yellow,830.3500
3,green,508.5988
4,none,343.5368


Step 3.4: Event Type & Quality Metrics

14.  Count of reviewed vs automatic earthquakes (status).

In [76]:
query = """
SELECT properties_status, count(*) AS count
FROM earthquake
GROUP BY properties_status
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,properties_status,count
0,reviewed,37251


15.  Count by earthquake type (type).

In [77]:
query = """
SELECT properties_type, count(*) AS count
FROM earthquake
GROUP BY properties_type
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,properties_type,count
0,earthquake,37241
1,mine collapse,1
2,volcanic eruption,9


16.  Number of earthquakes by data type (types).

In [78]:
import pandas as pd
import numpy as np

# fetch types column
cursor.execute("SELECT properties_types FROM earthquake;")
rows = cursor.fetchall()   # list of tuples like [('origin,phase-data',), (None,), ...]

# convert to Series safely
df_types = pd.DataFrame(rows, columns=['properties_types'])

# normalize to string, remove leading commas/spaces, turn empty -> NaN
s = (df_types['properties_types']
     .fillna('')                    # turn None -> ''
     .astype(str)                   # ensure string
     .str.strip()                   # trim whitespace
     .str.lstrip(',')               # remove leading commas only
     .replace('', np.nan)           # empty -> NaN
)

# drop missing and split/explode
df_exploded = (s.dropna()
                 .str.split(',', expand=False)
                 .explode()
                 .str.strip()          # clean each item
                 .to_frame(name='type') )

# count
type_counts = df_exploded['type'].value_counts()
print(type_counts)


type
origin                    37251
phase-data                37251
dyfi                       7414
moment-tensor              5353
shakemap                   3954
losspager                  3546
internal-moment-tensor     2844
internal-origin            1525
earthquake-name            1137
ground-failure              721
impact-text                 638
impact-link                 456
general-text                129
oaf                          89
nearby-cities                78
associate                    75
scitech-link                 74
finite-fault                 64
shake-alert                  63
focal-mechanism              51
groundmotion-packet          27
general-link                 18
trump-origin                 15
trump-shakemap                6
general-header                5
losspager-admin               5
disassociate                  4
significance                  3
trump-ground-failure          3
trump-losspager               3
poster                        2
tru

17.  Average RMS and gap per continent.

In [79]:
query = """
SELECT 
    continent,
    AVG(properties_rms) AS avg_rms,
    AVG(properties_gap) AS avg_gap
FROM earthquake
GROUP BY continent
ORDER BY continent;
"""


cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,continent,avg_rms,avg_gap
0,Africa,0.686997,81.345609
1,Asia,0.720656,82.629794
2,Europe,0.717081,81.872912
3,North America,0.799186,110.745278
4,Oceania,0.738070,93.813909
5,South America,0.779890,83.854235
6,Unknown,0.705094,92.908704


18.  Events with high station coverage (nst > threshold).


In [80]:
query = f"""
SELECT 
    id,
    magnitude,
    place,
    time,
    properties_nst AS nst,
    properties_magType,
    depth_km,
    country,
    continent
FROM earthquake
WHERE properties_nst > 50
ORDER BY properties_nst DESC limit 10;
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,id,magnitude,place,time,nst,properties_magType,depth_km,country,continent
0,us6000m12f,5.4,"11 km w of anamizu, japan",2024-01-02 01:17:32,619.0,mww,6.000,japan,Asia
1,usd001097k,5.5,"49 km wnw of san antonio de los cobres, argentina",2023-12-11 18:36:00,466.0,mww,191.769,argentina,South America
2,us7000ilwt,6.0,north pacific ocean,2022-11-02 04:53:13,452.0,mww,10.000,ocean,Unknown
3,us6000i6p3,5.1,kermadec islands region,2022-07-29 22:41:00,444.0,mb,532.015,region,Unknown
4,us6000len8,6.3,"24 km nnw of herāt, afghanistan",2023-10-11 00:41:56,423.0,mww,8.000,afghanistan,Asia
5,us7000ll46,5.8,"115 km sse of vilyuchinsk, russia",2023-12-23 17:48:05,420.0,mww,41.862,russia,Europe
6,us7000kg30,7.2,"2023 sand point, alaska earthquake",2023-07-16 06:48:21,410.0,mww,25.000,alaska earthquake,Unknown
7,us6000nqd8,5.8,"117 km e of ‘ohonua, tonga",2024-09-07 22:39:08,406.0,mww,10.000,tonga,Oceania
8,us7000myfa,7.1,"106 km wsw of sangay, philippines",2024-07-11 02:13:19,401.0,mww,639.503,philippines,Asia
9,us7000l6k4,5.9,"143 km e of ust’-kamchatsk staryy, russia",2023-10-26 16:05:13,388.0,mww,10.000,russia,Europe


Step 3.5: Tsunamis & Alerts

19.  Number of tsunamis triggered per year.

In [81]:
query = """
SELECT 
    year,
    COUNT(*) AS total_events,
    SUM(CASE WHEN properties_tsunami = 1 THEN 1 ELSE 0 END) AS tsunami_events
FROM earthquake
GROUP BY year
ORDER BY year;
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,year,total_events,tsunami_events
0,2020,6494,98
1,2021,8943,89
2,2022,7766,100
3,2023,7651,88
4,2024,6397,81


20.  Count earthquakes by alert levels (red, orange, etc.).

In [82]:
query = """
SELECT 
    COALESCE(properties_alert, 'no_alert') AS alert_level,
    COUNT(*) AS count_alerts
FROM earthquake
GROUP BY alert_level
ORDER BY count_alerts DESC;
"""
cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,alert_level,count_alerts
0,none,33705
1,green,3387
2,yellow,120
3,orange,22
4,red,17


Step 3.5: Seismic Pattern & Trends Analysis.

21.Find the top 5 countries with the highest average magnitude of earthquakes in the past 5 years    

In [83]:
query = """
SELECT country, avg(magnitude) as avg_mag from earthquake
group by country
order by avg_mag 
limit 5;
"""
cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,country,avg_mag
0,montenegro,4.5
1,kansas,4.5
2,israel,4.5
3,washington,4.5
4,saint pierre and miquelon,4.5


22.Find countries that have experienced both shallow and deep earthquakes within the same month.

In [84]:
query = """
SELECT DISTINCT e1.country, 
       e1.year,
       e1.month
FROM earthquake e1
WHERE e1.depth_category = 'Shallow'
AND EXISTS (
        SELECT 1
        FROM earthquake e2
        WHERE e2.country = e1.country
          AND e2.year = e1.year
          AND e2.month = e1.month
          AND e2.depth_category = 'Deep'
)
UNION
SELECT DISTINCT e2.country, 
       e2.year,
       e2.month
FROM earthquake e2
WHERE e2.depth_category = 'Deep'
AND EXISTS (
        SELECT 1
        FROM earthquake e3
        WHERE e3.country = e2.country
          AND e3.year = e2.year
          AND e3.month = e2.month
          AND e3.depth_category = 'Shallow'
);

"""
cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,country,year,month
0,japan,2020,1
1,region,2020,1
2,solomon islands,2020,1
3,alaska,2020,1
4,philippines,2020,1
...,...,...,...
1409,tajikistan,2024,12
1410,northern mariana islands,2024,12
1411,nicaragua,2024,12
1412,new zealand,2024,12


23.Compute the year-over-year growth rate in the total number of earthquakes globally.

In [ ]:
# YoY Growth (%)=(ThisYear−LastYear)/LastYear​×100

query = """
WITH yearly_counts AS (
    SELECT 
        year,
        COUNT(*) AS total_quakes
    FROM earthquake
    GROUP BY year
)
SELECT 
    y1.year,
    y1.total_quakes,
    y2.total_quakes AS previous_year_total,
    ROUND(
        ((y1.total_quakes - y2.total_quakes) / y2.total_quakes) * 100,
        2
    ) AS yoy_growth_percentage
FROM yearly_counts y1
LEFT JOIN yearly_counts y2
    ON y1.year = y2.year + 1
ORDER BY y1.year;
"""
cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,year,total_quakes,previous_year_total,yoy_growth_percentage
0,2020,6494,NaN,None
1,2021,8943,6494.0,37.71
2,2022,7766,8943.0,-13.16
3,2023,7651,7766.0,-1.48
4,2024,6397,7651.0,-16.39


24. List the 3 most seismically active regions by combining both frequency and average magnitude.


In [89]:
query = """
SELECT 
    country AS region,
    COUNT(*) AS total_events,
    ROUND(AVG(magnitude), 3) AS avg_magnitude,
    ROUND(COUNT(*) * AVG(magnitude), 3) AS activity_score
FROM earthquake
GROUP BY country
ORDER BY activity_score DESC
LIMIT 3;

"""
cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,region,total_events,avg_magnitude,activity_score
0,region,4648,4.820,22404.39
1,indonesia,3465,4.789,16592.60
2,ridge,2439,4.789,11679.60


Step 3.5: Depth, Location & Distance-Based  Analysis.

25. For each country, calculate the average depth of earthquakes within ±5° latitude range of the equator.

In [ ]:
query = """
SELECT 
    country,
    ROUND(AVG(depth_km), 3) AS avg_depth_equatorial
FROM earthquake
WHERE latitude BETWEEN -5 AND 5
GROUP BY country
ORDER BY avg_depth_equatorial DESC;
"""
cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,country,avg_depth_equatorial
0,philippines,93.861
1,sea,79.632
2,peru,68.539
3,papua new guinea,53.956
4,indonesia,51.788
5,ecuador,46.151
6,colombia,43.336
7,congo-uganda,14.775
8,sumatra,12.250
9,region,11.977


26. Identify countries having the highest ratio of shallow to deep earthquakes.

In [93]:
query="""SELECT
    country,
    SUM(CASE WHEN depth_category = 'Shallow' THEN 1 ELSE 0 END) AS shallow_count,
    SUM(CASE WHEN depth_category = 'Deep' THEN 1 ELSE 0 END) AS deep_count,
    ROUND(
        SUM(CASE WHEN depth_category = 'Shallow' THEN 1 ELSE 0 END) /
        NULLIF(SUM(CASE WHEN depth_category = 'Deep' THEN 1 ELSE 0 END), 0),
        3
    ) AS shallow_to_deep_ratio
FROM earthquake
GROUP BY country
HAVING deep_count > 0   -- only countries having at least 1 deep earthquake
ORDER BY shallow_to_deep_ratio DESC;
"""
cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,country,shallow_count,deep_count,shallow_to_deep_ratio
0,panama,180,1,180.000
1,turkey,315,2,157.500
2,micronesia,114,1,114.000
3,china,559,5,111.800
4,pakistan,74,1,74.000
...,...,...,...,...
57,dominica,0,3,0.000
58,malaysia,0,2,0.000
59,okhotsk,0,5,0.000
60,grenada,0,3,0.000


27. Find the average magnitude difference between earthquakes with tsunami alerts and those without.

In [95]:
query = """
SELECT
    AVG(CASE WHEN properties_tsunami = 1 THEN magnitude END) -
    AVG(CASE WHEN properties_tsunami = 0 THEN magnitude END)
    AS avg_magnitude_difference
FROM earthquake;
"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype


,avg_magnitude_difference
0,1.029576


28. Using the gap and rms columns, identify events with the lowest data reliability (highest average error margins).

In [100]:
query = """
SELECT
    id,
    place,
    magnitude,
    properties_rms AS rms,
    properties_gap AS gap,
    (properties_rms + properties_gap)/2 AS error_score
FROM earthquake
ORDER BY error_score DESC
LIMIT 10;

"""

cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,id,place,magnitude,rms,gap,error_score
0,pr2020283000,"85 km ene of saint croix, u.s. virgin islands",4.50,0.49,348.0,174.245
1,nc73351825,"73km w of petrolia, ca",4.94,0.19,321.0,160.595
2,pr2020331007,"105 km n of charlotte amalie, u.s. virgin islands",4.71,0.38,317.0,158.690
3,us6000a6gy,south sandwich islands region,4.60,0.54,291.0,145.770
4,us7000dfhu,"248 km ene of gisborne, new zealand",4.60,0.90,290.0,145.450
5,us7000emi2,chagos archipelago region,4.60,0.95,283.0,141.975
6,nc73541781,"76km w of petrolia, ca",4.69,0.28,282.0,141.140
7,pr2020035000,"61 km ne of miches, dominican republic",4.50,0.88,281.0,140.940
8,us6000lp5x,west of the galapagos islands,4.70,0.60,281.0,140.800
9,us7000dfgb,"294 km ene of gisborne, new zealand",4.70,0.63,280.0,140.315


29. Find pairs of consecutive earthquakes (by time) that occurred within 50 km of each other and within 1 hour.

In [ ]:
query="""SELECT 
    e1.id AS eq1_id,
    e2.id AS eq2_id,
    e1.time AS eq1_time,
    e2.time AS eq2_time,

    TIMESTAMPDIFF(MINUTE, e1.time, e2.time) AS time_diff_minutes,

    6371 * ACOS(
        COS(RADIANS(e1.latitude)) * COS(RADIANS(e2.latitude)) *
        COS(RADIANS(e2.longitude) - RADIANS(e1.longitude)) +
        SIN(RADIANS(e1.latitude)) * SIN(RADIANS(e2.latitude))
    ) AS distance_km

FROM earthquake e1
JOIN earthquake e2
    ON e2.time > e1.time
    AND e2.time <= DATE_ADD(e1.time, INTERVAL 1 HOUR)   

WHERE NOT EXISTS (
    SELECT 1 
    FROM earthquake e3
    WHERE e3.time > e1.time 
      AND e3.time < e2.time
)

HAVING distance_km <= 50
ORDER BY e1.time;
"""
cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

30. Determine the regions with the highest frequency of deep-focus earthquakes (depth > 300 km).

In [112]:
query="""SELECT 
    country,
    COUNT(*) AS deep_focus_count
FROM earthquake
WHERE depth_km > 300
GROUP BY country
ORDER BY deep_focus_count DESC;
"""
cursor.execute(query)
df_magtype = pd.DataFrame(cursor.fetchall(), columns=[col[0] for col in cursor.description])
df_magtype

,country,deep_focus_count
0,islands,505
1,region,373
2,fiji,227
3,tonga,106
4,indonesia,74
5,japan region,49
6,timor leste,47
7,wallis and futuna,29
8,japan,28
9,papua new guinea,22
